In [1]:
"""
    coeficientes_hermite(x::Vector, y::Vector, dy::Vector)

Calcula los coeficientes del polinomio de interpolación de Hermite.
Basado en el algoritmo de diferencias divididas.

Argumentos:
- `x`: Vector de nodos x_0, ..., x_n
- `y`: Vector de valores de la función f(x_0), ..., f(x_n)
- `dy`: Vector de derivadas f'(x_0), ..., f'(x_n)

Retorna:
- `z`: Vector de nodos duplicados.
- `Q`: La matriz completa de diferencias divididas (aunque solo la diagonal es necesaria para el polinomio).
"""
function coeficientes_hermite(x::Vector{T}, y::Vector{T}, dy::Vector{T}) where T <: AbstractFloat
    n = length(x)
    if length(y) != n || length(dy) != n
        error("Los vectores x, y, y dy deben tener la misma longitud.")
    end

    # Total de puntos para Hermite es 2n
    # En el libro usan índices hasta 2n+1 (que son 2n puntos en total si n es conteo de 0 a n)
    # Aquí n es la longitud del vector, así que tendremos 2*n puntos en la tabla z.
    m = 2 * n 
    z = zeros(T, m)
    Q = zeros(T, m, m)

    # --- PASO 1 y 2: Inicialización y Primeras Diferencias ---
    # Nota: Ajustamos índices de 0-based (libro) a 1-based (Julia)
    # Libro i=0...n  => Julia i=1...n
    
    for i in 1:n
        # Mapeo de índices:
        # Libro: 2i     -> Julia: 2i - 1
        # Libro: 2i+1   -> Julia: 2i
        
        k = 2*i - 1 
        
        # Paso 2 del algoritmo
        z[k]     = x[i]
        z[k+1]   = x[i]
        
        Q[k, 1]   = y[i]
        Q[k+1, 1] = y[i]
        
        Q[k+1, 2] = dy[i] # Primera derivada colocada en la columna de 1ras diferencias
        
        # Paso 3 del algoritmo: Diferencias divididas para nodos distintos
        # Si i != 0 (en libro) -> Si i > 1 (en Julia)
        if i > 1
            # Q[2i, 1] en libro es Q[k, 2] en Julia (Columna 2 es 1ra diferencia)
            # Numerador: Q[2i, 0] - Q[2i-1, 0] -> Q[k, 1] - Q[k-1, 1]
            # Denominador: z[2i] - z[2i-1]     -> z[k] - z[k-1]
            Q[k, 2] = (Q[k, 1] - Q[k-1, 1]) / (z[k] - z[k-1])
        end
    end

    # --- PASO 4: Resto de la tabla de diferencias divididas ---
    # Libro: Para i = 2, ..., 2n+1
    # Julia: Para i = 3, ..., 2n (Empezamos en 3 porque las cols 1 y 2 ya tienen datos base)
    
    for i in 3:m
        # Libro: para j = 2, ..., i
        # Julia: j representa la columna.
        # Las columnas 1 y 2 ya están llenas. Queremos llenar desde la columna 3 (2das diferencias)
        # hasta la diagonal i.
        for j in 3:i
            # Fórmula: Q[i,j] = (Q[i, j-1] - Q[i-1, j-1]) / (z[i] - z[i - (j-1)])
            # Ajuste de índice en denominador z: z[i] - z[i-j+1]
            Q[i, j] = (Q[i, j-1] - Q[i-1, j-1]) / (z[i] - z[i-j+1])
        end
    end

    # --- PASO 5: Salida ---
    # Retornamos z y la diagonal de Q (que son los coeficientes del polinomio)
    coeficientes = [Q[i, i] for i in 1:m]
    
    return z, coeficientes
end

"""
    evaluar_hermite(z, coefs, valor_x)

Evalúa el polinomio de Hermite en un punto específico usando el método de Horner anidado.
H(x) = Q_0 + Q_1(x-z_0) + Q_2(x-z_0)(x-z_1) + ...
"""
function evaluar_hermite(z::Vector, coefs::Vector, valor_x::Number)
    n = length(coefs)
    resultado = coefs[n]
    
    # Evaluación de atrás hacia adelante (Horner)
    for i in (n-1):-1:1
        resultado = resultado * (valor_x - z[i]) + coefs[i]
    end
    
    return resultado
end

evaluar_hermite

In [2]:
# 1. Definir datos de prueba
x_puntos = [0.0, 1.0, 2.0]        # x_0, x_1, x_2
y_puntos = exp.(x_puntos)         # f(x)
dy_puntos = exp.(x_puntos)        # f'(x)

# 2. Obtener coeficientes y nodos z
println("Calculando coeficientes de Hermite...")
z_nodos, coefs = coeficientes_hermite(x_puntos, y_puntos, dy_puntos)

# Mostrar la tabla diagonal (Coeficientes)
println("\nCoeficientes del Polinomio (Diagonal de Q):")
println(coefs)

# 3. Evaluar en un punto intermedio, por ejemplo x = 0.5
val_exacto = exp(0.5)
val_aprox = evaluar_hermite(z_nodos, coefs, 0.5)

println("\n--- Evaluación en x = 0.5 ---")
println("Valor exacto (e^0.5): $val_exacto")
println("Aproximación Hermite: $val_aprox")
println("Error absoluto:       $(abs(val_exacto - val_aprox))")

Calculando coeficientes de Hermite...

Coeficientes del Polinomio (Diagonal de Q):
[1.0, 1.0, 0.7182818284590451, 0.2817181715409549, 0.09726402473266271, 0.023753778993719554]

--- Evaluación en x = 0.5 ---
Valor exacto (e^0.5): 1.6487212707001282
Aproximación Hermite: 1.6482077704372722
Error absoluto:       0.0005135002628560148
